# M5 · Eficiencia: cuantización a 4 bits (Camilo)

Regla de decisión (escrita antes de correr): `m5/eficiencia.md`.

**Antes de correr, en Kaggle:**
- *Settings → Accelerator* → GPU T4 (x1 o x2; el código usa solo la primera GPU).
- *Settings → Internet* → On.
- La rama `camilo` debe estar subida a GitHub (este notebook la clona).

Cada configuración corre en un proceso nuevo (`!python`), para que la memoria medida sea la suya. Cada corrida completa tarda unos 15-20 min.

## 1 · Entorno

In [ ]:
import os
RAMA = "camilo"
if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia
!git log -1 --oneline

In [ ]:
%pip install -q -r requirements.txt
%pip install -q bitsandbytes
# Mismo arreglo que en los notebooks de M2-M5: torchao viejo hace tronar a peft al importarse.
!pip uninstall -y -q torchao

In [ ]:
import torch, transformers
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__)
import bitsandbytes
print("bitsandbytes", bitsandbytes.__version__)

## 2 · Prueba de humo (2 casos por corrida)
Confirma que las dos configuraciones cargan y miden antes de gastar 15-20 min cada una. Los archivos que deja se sobrescriben en la corrida completa.

In [ ]:
!python m5/eficiencia/medir.py --config referencia --limite 2

In [ ]:
!python m5/eficiencia/medir.py --config 4bit --limite 2

## 3 · Corrida de referencia (sin cuantizar)

In [ ]:
!python m5/eficiencia/medir.py --config referencia

## 4 · Corrida a 4 bits (cerebro + generador)

In [ ]:
!python m5/eficiencia/medir.py --config 4bit

## 5 · Tabla y veredicto según la regla

In [ ]:
!python m5/eficiencia/medir.py --tabla

## 6 · Opcional: solo si 4 bits incumple la regla
Separan qué componente rompe la calidad (`4bit_cerebro`, `4bit_generador`) y si la diferencia viene del dtype y no de la cuantización (`fp16`). Si 4 bits cumple la regla, no hace falta correrlas.

In [ ]:
!python m5/eficiencia/medir.py --config 4bit_cerebro

In [ ]:
!python m5/eficiencia/medir.py --config 4bit_generador

In [ ]:
!python m5/eficiencia/medir.py --config fp16

## 7 · Empaquetar resultados
Descarga `m5_eficiencia.zip` desde *Output* y extráelo en la raíz del repo (sus rutas empiezan por `m5/eficiencia/`).

In [ ]:
import os, zipfile
with zipfile.ZipFile("/kaggle/working/m5_eficiencia.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for c in ["referencia", "fp16", "4bit", "4bit_cerebro", "4bit_generador"]:
        base = f"m5/eficiencia/{c}"
        if os.path.isdir(base):
            for d, _, fs in os.walk(base):
                for f in fs:
                    z.write(os.path.join(d, f))
print(zipfile.ZipFile("/kaggle/working/m5_eficiencia.zip").namelist())